# Week 11 — Monday: Logistic Regression

**DATA 202 · Calvin University**

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/11/images/pumpkins.jpg" width="480" alt="A pile of small orange pumpkins on a bale of straw.">

In Turkey, roasted pumpkin seeds are a favorite snack. Two varieties: **Çerçevelik** and **Ürgüp Sivrisi** ("the pointy one from Ürgüp"). A team of researchers photographed 2,500 seeds and measured each seed's shape from its photo.

> 🎯 **Can we tell the variety from the seed's shape, and how sure can we be?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~5 min | 1 · A line for a yes/no label |
| ~12 min | 2 · The S-curve: probabilities (SLO 11A) |
| ~8 min | 3 · Two columns, one straight boundary (SLO 11A) |
| ~8 min | 4 · All 12 columns, and the C knob (SLO 11A) |
| ~5 min | 5 · More than two varieties (SLO 11A) |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

<sub>Data: Koklu, Sarigil & Ozbek (2021), *Genetic Resources and Crop Evolution* 68(7), 2713–2726. Photo: Unsplash</sub>

---
## 1 · A Line for a Yes/No Label · ~5 min

One row = one seed. Lengths are in pixels, measured from the photo.

* `length`, `width`: the seed's long and short axes
* `aspect_ratio` = length ÷ width: big = long and pointy
* `roundness`, `compactness`, … : other shape measures (12 columns in all)

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

seeds = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/pumpkin.csv")
X = seeds.drop(columns="variety")
y = seeds["variety"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)
print(len(X_train), "training seeds |", len(X_test), "test seeds")
seeds.head()

In [ ]:
fig = px.strip(x=X_train["aspect_ratio"], y=y_train, labels={"x": "aspect ratio (length ÷ width)", "y": ""},
               title="1,875 Training Seeds: Aspect Ratio by Variety")
fig.show()

> 🎯 **Predict First:** a seed with aspect ratio 2.0. Which variety? How sure are you?

Week 10's tool predicts numbers. Turn the label into a number, 1 = Ürgüp Sivrisi and 0 = Çerçevelik, and fit a line:

In [ ]:
is_urgup = (y_train == "Ürgüp Sivrisi").astype(int)       # 1 = Ürgüp Sivrisi, 0 = Çerçevelik
line = LinearRegression().fit(X_train[["aspect_ratio"]], is_urgup)
guess = line.predict(X_train[["aspect_ratio"]])
print(f"the line's guesses run from {guess.min():.2f} to {guess.max():.2f}")

> 💬 **Question:** If we read the line's guess as "the chance this seed is Ürgüp Sivrisi", what does −0.54 mean? And 1.73?

<details><summary>Answer</summary>

* A chance of −54% or 173%: nonsense. A chance must stay between 0 and 1
* A straight line always leaves that range eventually. We need a curve that bends to stay inside
</details>

---
## 2 · The S-Curve: Probabilities (SLO 11A) · ~12 min

**Logistic regression** works in two steps:

1. A **score**, from a line, as in Week 10: $z = b_0 + b_1 \times \text{aspect ratio}$
2. **Squeeze** the score into a probability with the S-shaped *logistic* (sigmoid) curve: $p = \dfrac{1}{1 + e^{-z}}$

| score $z$ | −5 | −2 | 0 | 2 | 5 |
|---|---|---|---|---|---|
| probability $p$ | 0.01 | 0.12 | 0.50 | 0.88 | 0.99 |

It predicts the variety with $p$ above 0.5.

In [ ]:
logit = LogisticRegression().fit(X_train[["aspect_ratio"]], y_train)
print(f"score z = {logit.intercept_[0]:.2f} + {logit.coef_[0][0]:.2f} × aspect ratio")

ratios = pd.DataFrame({"aspect_ratio": np.linspace(1.1, 3.2, 200)})
fig = px.scatter(x=X_train["aspect_ratio"], y=is_urgup, opacity=0.15,
                 labels={"x": "aspect ratio (length ÷ width)", "y": "1 = Ürgüp Sivrisi, 0 = Çerçevelik"},
                 title="Logistic Regression: the Probability of Ürgüp Sivrisi")
fig.add_scatter(x=ratios["aspect_ratio"], y=logit.predict_proba(ratios)[:, 1], name="P(Ürgüp Sivrisi)")
fig.show()

> 📄 **Handout 1 — The S-curve.** Mark the 50/50 point. Shade the side where the model says Ürgüp Sivrisi. For seeds A, B and C: which variety, and how sure?

In [ ]:
three = pd.DataFrame({"aspect_ratio": [1.8, 2.0, 2.4]}, index=["A", "B", "C"])
# predict_proba gives one column per variety, in the order of logit.classes_: Çerçevelik, then Ürgüp Sivrisi
three["P(Ürgüp Sivrisi)"] = logit.predict_proba(three[["aspect_ratio"]])[:, 1].round(2)
three["predicted"] = logit.predict(three[["aspect_ratio"]])
three

<details><summary>Answer</summary>

* **50/50 at aspect ratio 2.03**, where the score is 0: 17.62 ÷ 8.68
* **A** (1.8): 12% → Çerçevelik, fairly sure · **B** (2.0): 44% → Çerçevelik, almost a coin flip · **C** (2.4): 96% → Ürgüp Sivrisi, very sure
* **Reading the slope:** +8.68 per unit of aspect ratio, so +0.868 per 0.1. Each +0.1 multiplies the **odds** $p/(1-p)$ by $e^{0.868} \approx$ **2.4**. Positive slope → longer seeds lean Ürgüp Sivrisi
* One column already gets **86.8%** right in 5-fold CV
</details>

> "Organizations apply labeling and scoring methods to slice people into groups and ranks suited to their particular ends. […] Instead they tend to be behavioral and probabilistic, predicting the likelihood that people will do or like certain kinds of things, or the position they may reach on a particular scale."
> — Fourcade & Healy, *The Ordinal Society*, ch. 3

> 💬 **Question:** Seed B is 44% Ürgüp Sivrisi and gets the same label as a seed at 2%. Now replace the seed with a person, scored on "likely to repay a loan." What is lost when 44% becomes a plain *no*? And who chose 50% as the line?

<details><summary>Answer</summary>

* The label hides how unsure the model was: 44% and 2% both become *no*
* The cutoff is a choice, not a fact of the data. A bank can move it to suit "its particular ends" (Week 12 moves it on purpose)
* A seed doesn't mind being misread. A person's life chances can turn on a probability they never see
</details>

---
## 3 · Two Columns, One Straight Boundary (SLO 11A) · ~8 min

Instead of the ratio, give the model length and width separately.

In [ ]:
fig = px.scatter(x=X_train["width"], y=X_train["length"], color=y_train, opacity=0.5,
                 labels={"x": "width (pixels)", "y": "length (pixels)", "color": "variety"},
                 title="Training Seeds: Length vs. Width")
fig.show()

> 📄 **Handout 2 — Draw the boundary.** Draw the straight line you would use to separate the two varieties. Circle a region where no straight line could get every seed right.

In [ ]:
logit2 = LogisticRegression().fit(X_train[["length", "width"]], y_train)
b0 = logit2.intercept_[0]
b_length, b_width = logit2.coef_[0]

# The 50/50 seeds have score 0: b0 + b_length × length + b_width × width = 0. Solved for length:
widths = np.linspace(150, 310, 50)
fig.add_scatter(x=widths, y=-(b0 + b_width * widths) / b_length, mode="lines", name="50/50 boundary")
fig.show()
print(f"boundary: length = {-b0 / b_length:.0f} + {-b_width / b_length:.2f} × width")
print(f"5-fold CV accuracy: {cross_val_score(LogisticRegression(), X_train[['length', 'width']], y_train, cv=5).mean():.3f}")

<details><summary>Answer</summary>

* The boundary is a **straight line**: length = 48 + 1.81 × width. Longer than about 1.8–2 widths → Ürgüp Sivrisi. The model rediscovered the aspect ratio
* Logistic regression's boundary is always straight (a flat plane with more columns): the score is a line
* Where the colors overlap, no straight line can win. CV accuracy: **87.0%**, about the same as the ratio alone
</details>

---
## 4 · All 12 Columns, and the C Knob (SLO 11A) · ~8 min

* With 12 columns, **scale first**: `LogisticRegression` has Week 10's **Ridge penalty built in**
* Its knob is `C` = 1 ÷ alpha. **Small C = big penalty**
* Choose C with cross-validation. (`logisticregression__C` = the `C` of the pipeline step named `logisticregression`)

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=10_000))
Cs = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000]
search = GridSearchCV(model, {"logisticregression__C": Cs}, cv=5, return_train_score=True)
search.fit(X_train, y_train)

results = pd.DataFrame({"C": Cs,
                        "folds it learned from": search.cv_results_["mean_train_score"],
                        "held-out fold": search.cv_results_["mean_test_score"]})
fig = px.line(results, x="C", y=["folds it learned from", "held-out fold"], markers=True, log_x=True,
              labels={"value": "accuracy", "variable": "scored on"}, title="Logistic Regression: Choosing C by 5-Fold CV")
fig.show()

> 📄 **Handout 3 — Choose C.** Circle the best C. Label the underfitting side. Is there an overfitting side?

<details><summary>Answer</summary>

* **Small C** (0.0001): 83.3% on both: the penalty flattens every slope: **underfitting**
* **Best: C = 100**, **88.9%** held-out
* **No overfitting side:** the two lines stay together. 12 columns are too little freedom to memorize 1,875 seeds
* Here a **larger** C means a **more complex** model: the opposite direction from Ridge's alpha
</details>

In [ ]:
best = search.best_estimator_
pd.Series(best[-1].coef_[0], index=X.columns, name="slope (per standard deviation)").round(1).sort_values()

<details><summary>Answer</summary>

* `equiv_diameter` −23.2 against `area` +7.3 and `convex_area` +5.5: three near-copies of one measurement (correlation 1.00) pulling against each other. Week 10's tug of war again
* So read single slopes with care. The predictions are trustworthy; the story behind each slope less so
</details>

> 📌 **One split:** 88.9% comes from 5 folds of these 1,875 seeds. The test set stays locked until Wednesday's final comparison.

---
## 5 · More Than Two Varieties (SLO 11A) · ~5 min

Suppose the seed company sold a third variety. `LogisticRegression` would:

1. Give **each variety its own score** $z$ (its own line)
2. Turn the scores into probabilities that **add up to 1** (the *softmax*: $e^{z}$ for each, divided by the total)
3. Predict the variety with the **largest** probability

Same code, `LogisticRegression().fit(X, y)`, with three names in `y`. For one seed:

In [ ]:
scores = pd.Series({"Çerçevelik": 1.2, "Ürgüp Sivrisi": 0.4, "a third variety": -2.0})
probabilities = np.exp(scores) / np.exp(scores).sum()       # softmax
probabilities.round(2)

<details><summary>Answer</summary>

* 67% Çerçevelik, 30% Ürgüp Sivrisi, 3% the third variety → **Çerçevelik**
* With two varieties, this is exactly Part 2's S-curve
</details>

---
### Wrap-up

1. **Logistic regression** = a line's score, squeezed by the S-curve into a probability between 0 and 1
2. Predict the class above 50%. The 50/50 boundary is **straight**
3. A slope's sign says which way the odds move; $e^{\text{slope}}$ says by how much
4. `C` is a built-in Ridge penalty, backwards: small C = simpler. Choose it with cross-validation 📌
5. More than two classes: one score per class, softmax, pick the largest

**Wednesday:** a straight boundary got 88.9%. Can a forest of trees, with its bendy boundaries, beat it?